# Bender / iseg NHS testen

COM11: negatives Modul. COM12: positives Modul.

Zellen der Reihe nach ausführen. Standardmäßig wird nur gelesen. Kanalzuordnung: p0/p1 = COM12, Kanäle 0/1; n0/n1 = COM11, Kanäle 0/1. Python >= 3.10 und ein Jupyter-Python-Kernel sind erforderlich. Andere Programme müssen die Ports freigeben.

**Das Schließen einer Verbindung schaltet Hochspannung nicht aus.**

Die gespeicherten Ausgaben dokumentieren frühere Hardwareläufe. Für einen neuen Versuch in der Konfigurationszelle die Freigabe des gewünschten Abschnitts setzen. Alle fünf Freigaben stehen standardmäßig auf `False`.

Kommunikationsparameter und Ports kommen aus `config/devices.example.json`. Für lokale Einstellungen eine Kopie `devices.local.json` erstellen und `CONFIG_PATH` in der folgenden Zelle anpassen. Der Lader prüft die Einstellungen ohne Hardwarezugriff. `with bender_device` öffnet und schließt die Verbindung pro Testzelle. Nach Änderungen die Konfigurationszelle erneut ausführen.


In [1]:
from pathlib import Path
import logging
import sys
import time
from dataclasses import asdict
from IPython.display import display

candidates = [Path.cwd(), *Path.cwd().parents, Path.cwd() / "LAPIS-Control-Suite"]
project_root = next((p.resolve() for p in candidates
                     if (p / "pyproject.toml").is_file()
                     and (p / "src/lapis_control").is_dir()), None)
if project_root is None:
    raise FileNotFoundError("Open this notebook from the LAPIS project checkout.")
src_dir = project_root / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))
driver_dir = src_dir / "lapis_control/devices/bender"

# Uncomment once if this kernel lacks the project dependencies:
# %pip install -e "{project_root}[notebook]"

from serial.tools import list_ports
from lapis_control.configuration import load_devices
from lapis_control.devices.bender.iseg_nhs import ChannelStatus

logging.getLogger(ChannelStatus.__module__).setLevel(logging.INFO)

# Select devices.local.json explicitly when using a local setup.
CONFIG_PATH = project_root / "config/devices.example.json"
bender_device = load_devices(CONFIG_PATH)["bender"]  # No hardware access.
POSITIVE_PORT = bender_device.positive.port
NEGATIVE_PORT = bender_device.negative.port
MAX_VOLTAGE_V = bender_device.positive.max_voltage
ALLOW_N0_DIAGNOSTIC = False
ALLOW_FOUR_CHANNEL_TEST = False
ALLOW_WRITES = False
ALLOW_HV_ON = False
ALLOW_P0_TEST = False

def show_channels(bender):
    rows = []
    for name, reading in bender.read_all().items():
        row = {"name": name, **asdict(reading)}
        row["status"] = f"0x{int(reading.status):04X} ({reading.status!r})"
        row["events"] = f"0x{int(reading.events):04X} ({reading.events!r})"
        rows.append(row)
    display(rows)


## 1. Verfügbare COM-Ports

In [2]:
for port in sorted(list_ports.comports(), key=lambda p: p.device):
    print(f"{port.device}: {port.description} | Seriennummer: {port.serial_number}")

COM10: Serielles USB-Gerät (COM10) | Seriennummer: 3853366F3432
COM11: USB Serial Port (COM11) | Seriennummer: A600C0IIA
COM12: USB Serial Port (COM12) | Seriennummer: AI05M8IUA
COM13: USB Serial Port (COM13) | Seriennummer: A9RFD6SEA


## 1a. Wiederholter USB-Kommunikationstest (beide Module am Hub)

Diesen Test **statt Punkt 2** ausführen. Er benötigt nur pyserial und verwendet weder Bender noch den NHS-Treiber. Er findet beide Module anhand ihrer USB-Seriennummern, auch wenn sich die COM-Ports nach dem Umstecken geändert haben. Die Gerätekennung wird zusätzlich anhand der NHS-Geräte-Seriennummer geprüft.

Pro Modul laufen fünf Runden mit `*IDN?` und `:READ:FIRMWARE:NAME?`. `PROBE_MODE = "block"` testet den Versand einer ganzen Befehlszeile; mit `"byte_echo"` kannst du anschließend den zeichenweisen Echo-Handshake vergleichen. Jeder Fehler beendet die aktuelle Runde; die nächste beginnt mit einer neuen Verbindung und einem CR/LF-Zeilenabschluss für eventuell unvollständige **Leseabfragen**. Nicht direkt nach einem abgebrochenen Schreibbefehl verwenden. Es werden keine Sollwerte, Ausgänge, Ereignisse oder Echo-Einstellungen verändert.

Andere Programme müssen die Ports freigeben. Rohdaten, Fehler und Zusammenfassung erscheinen darunter; anschließend mit **Strg+S speichern**. Dies ist ein unabhängiger Python-Test, kein Aufruf des Herstellerprogramms iseg Terminal.

In [16]:
# Independent USB read test without the NHS/Bender driver.
# Only these two queries are allowed; no setpoint or output commands.
import time
import serial
from serial.tools import list_ports
from IPython.display import display

PROBE_ROUNDS = 5
PROBE_MODE = "block"  # Alternative: "byte_echo" (requires echo to be enabled).
PROBE_TIMEOUT_S = 2.0
PROBE_MODULES = [
    {"name": "Positiv", "usb_serial": "AI05M8IUA", "device_serial": "8000311"},
    {"name": "Negativ", "usb_serial": "A600C0IIA", "device_serial": "8000312"},
]
usb_probe_results = []
usb_probe_log = []


def probe_trace(port, direction, raw):
    line = f"{port} {direction} {raw!r}"
    usb_probe_log.append(line)
    print(line)


def probe_query(connection, command, expected_serial, mode):
    if command not in ("*IDN?", ":READ:FIRMWARE:NAME?"):
        raise ValueError("Dieser Test erlaubt ausschliesslich IDN- und Firmware-Leseabfragen.")
    if mode not in ("block", "byte_echo"):
        raise ValueError("PROBE_MODE muss block oder byte_echo sein.")
    # Do not assign timeout during a query transaction:
    # On Windows, pyserial reconfigures the entire port when timeout is assigned.
    wire = (command + "\r\n").encode("ascii")
    probe_trace(connection.port, "TX", wire)
    if mode == "byte_echo":
        for index, value in enumerate(wire):
            byte = bytes((value,))
            if connection.write(byte) != 1:
                raise RuntimeError("Unvollstaendig gesendetes Query-Zeichen.")
            echo = connection.read(1)
            if echo != byte:
                probe_trace(connection.port, f"ECHO[{index}], erwartet {byte!r}", echo)
                raise RuntimeError(f"Echo-Fehler bei Byte {index}: {byte!r} -> {echo!r}")
        print(f"{connection.port}: alle Zeichen-Echos korrekt")
    elif connection.write(wire) != len(wire):
        raise RuntimeError("Leseabfrage unvollstaendig gesendet.")

    deadline = time.monotonic() + PROBE_TIMEOUT_S
    while time.monotonic() < deadline:
        raw = connection.read_until(b"\r\n", 202)
        probe_trace(connection.port, "RX", raw)
        if not raw.endswith(b"\r\n") or len(raw) > 200:
            raise RuntimeError(f"Fehlende/abgeschnittene Antwort auf {command!r}")
        answer = raw[:-2].decode("ascii").strip()
        if not answer or answer == command:
            continue
        if command == "*IDN?":
            fields = [field.strip() for field in answer.split(",")]
            valid = (len(fields) >= 4 and "iseg" in fields[0].lower()
                     and fields[1].startswith("NHS") and fields[2] == expected_serial)
        else:
            valid = answer == "N06C2"
        if not valid:
            raise RuntimeError(f"Ungueltige Antwort oder beschaedigtes Echo: {answer!r}")
        return answer
    raise TimeoutError(f"Keine Antwort auf {command!r}")


if PROBE_MODE not in ("block", "byte_echo") or type(PROBE_ROUNDS) is not int or not 1 <= PROBE_ROUNDS <= 20:
    raise ValueError("Modus pruefen; PROBE_ROUNDS muss zwischen 1 und 20 liegen.")

available_ports = list(list_ports.comports())
print("Aktuelle Ports:")
for port in available_ports:
    print(f"  {port.device}: {port.serial_number!r} | {port.description}")

for module in PROBE_MODULES:
    matches = [port for port in available_ports if port.serial_number == module["usb_serial"]]
    if len(matches) != 1:
        message = f'{module["name"]}: {len(matches)} Ports mit USB-ID {module["usb_serial"]!r} gefunden'
        print(message)
        usb_probe_results.append({"module": module["name"], "ok": False, "error": message})
        continue
    port_name = matches[0].device
    print(f'\n=== {module["name"]}: {port_name}, Modus {PROBE_MODE} ===')
    for round_number in range(1, PROBE_ROUNDS + 1):
        row = {"module": module["name"], "port": port_name, "mode": PROBE_MODE,
               "round": round_number, "ok": False, "idn": None, "firmware": None,
               "failed_command": None, "error": None}
        started = time.monotonic()
        try:
            with serial.Serial(port=port_name, baudrate=9600, bytesize=8,
                               parity="N", stopbits=1, timeout=PROBE_TIMEOUT_S,
                               write_timeout=PROBE_TIMEOUT_S, xonxoff=False,
                               rtscts=False, dsrdtr=False) as connection:
                row["failed_command"] = "Lesezeile abschliessen"
                connection.reset_input_buffer()
                # Only for this diagnostic after earlier read queries:
                # Terminate any partial read query with CR/LF without resetting the device.
                # Each cycle starts with a fresh connection and a line boundary.
                connection.timeout = 0.25  # Configure before sending.
                if connection.write(b"\r\n") != 2:
                    raise RuntimeError("Zeilenabschluss unvollstaendig gesendet")
                residual = connection.read(256)
                probe_trace(port_name, "INIT-RX", residual)
                connection.timeout = PROBE_TIMEOUT_S
                row["failed_command"] = "*IDN?"
                row["idn"] = probe_query(connection, "*IDN?", module["device_serial"], PROBE_MODE)
                row["failed_command"] = ":READ:FIRMWARE:NAME?"
                row["firmware"] = probe_query(connection, ":READ:FIRMWARE:NAME?",
                                              module["device_serial"], PROBE_MODE)
                row["ok"] = True
                row["failed_command"] = None
        except Exception as exc:
            row["error"] = f"{type(exc).__name__}: {exc}"
        row["seconds"] = round(time.monotonic() - started, 3)
        usb_probe_results.append(row)
        label = "OK" if row["ok"] else row["error"]
        print(f'{port_name}, Runde {round_number}/{PROBE_ROUNDS}: {label}\n')
        time.sleep(0.2)

print("\nErgebnisse (jede Runde: IDN + Firmware):")
display(usb_probe_results)
for module in PROBE_MODULES:
    rows = [row for row in usb_probe_results if row["module"] == module["name"] and "round" in row]
    if rows:
        print(f'{module["name"]}: {sum(row["ok"] for row in rows)}/{len(rows)} Runden erfolgreich')
print("Alle Testverbindungen geschlossen. Ausgabe mit Strg+S im Notebook speichern.")


Aktuelle Ports:
  COM10: '3853366F3432' | Serielles USB-Gerät (COM10)
  COM11: 'A600C0IIA' | USB Serial Port (COM11)
  COM12: 'AI05M8IUA' | USB Serial Port (COM12)
  COM13: 'A9RFD6SEA' | USB Serial Port (COM13)

=== Positiv: COM12, Modus block ===
COM12 INIT-RX b'\r\n'
COM12 TX b'*IDN?\r\n'
COM12 RX b'*IDN?\r\n'
COM12 RX b'iseg Spezialelektronik GmbH,NHS 40 205,8000311,2.20\r\n'
COM12 TX b':READ:FIRMWARE:NAME?\r\n'
COM12 RX b':READ:FIRMWARE:NAME?\r\n'
COM12 RX b'N06C2\r\n'
COM12, Runde 1/5: OK

COM12 INIT-RX b'\r\n'
COM12 TX b'*IDN?\r\n'
COM12 RX b'*IDN?\r\n'
COM12 RX b'iseg Spezialelektronik GmbH,NHS 40 205,8000311,2.20\r\n'
COM12 TX b':READ:FIRMWARE:NAME?\r\n'
COM12 RX b':READ:FIRMWARE:NAME?\r\n'
COM12 RX b'N06C2\r\n'
COM12, Runde 2/5: OK

COM12 INIT-RX b'\r\n'
COM12 TX b'*IDN?\r\n'
COM12 RX b'*IDN?\r\n'
COM12 RX b'iseg Spezialelektronik GmbH,NHS 40 205,8000311,2.20\r\n'
COM12 TX b':READ:FIRMWARE:NAME?\r\n'
COM12 RX b':READ:FIRMWARE:NAME?\r\n'
COM12 RX b'N06C2\r\n'
COM12, Runde 3/5: 

[{'module': 'Positiv',
  'port': 'COM12',
  'mode': 'block',
  'round': 1,
  'ok': True,
  'idn': 'iseg Spezialelektronik GmbH,NHS 40 205,8000311,2.20',
  'firmware': 'N06C2',
  'failed_command': None,
  'error': None,
  'seconds': 0.511},
 {'module': 'Positiv',
  'port': 'COM12',
  'mode': 'block',
  'round': 2,
  'ok': True,
  'idn': 'iseg Spezialelektronik GmbH,NHS 40 205,8000311,2.20',
  'firmware': 'N06C2',
  'failed_command': None,
  'error': None,
  'seconds': 0.523},
 {'module': 'Positiv',
  'port': 'COM12',
  'mode': 'block',
  'round': 3,
  'ok': True,
  'idn': 'iseg Spezialelektronik GmbH,NHS 40 205,8000311,2.20',
  'firmware': 'N06C2',
  'failed_command': None,
  'error': None,
  'seconds': 0.496},
 {'module': 'Positiv',
  'port': 'COM12',
  'mode': 'block',
  'round': 4,
  'ok': True,
  'idn': 'iseg Spezialelektronik GmbH,NHS 40 205,8000311,2.20',
  'firmware': 'N06C2',
  'failed_command': None,
  'error': None,
  'seconds': 0.516},
 {'module': 'Positiv',
  'port': 'COM12'

Positiv: 5/5 Runden erfolgreich
Negativ: 5/5 Runden erfolgreich
Alle Testverbindungen geschlossen. Ausgabe mit Strg+S im Notebook speichern.


## 1b. Unabhängiger Windows-Test ohne Download

Dieser Test verwendet die in Windows vorhandene .NET-Schnittstelle über PowerShell. pyserial wird nur zum Finden der Ports benutzt, nicht zur Kommunikation. Das lokale Skript `tests/test_usb_dotnet.ps1` sendet ausschließlich Leseabfragen: Gerätekennung, Firmware und Status/Ereignisse von Kanal 0/1, fünfmal pro Modul. Es ändert keine Sollwerte, Ausgänge, Ereignisse oder Systemeinstellungen.

**Statt Punkt 2 ausführen**, während kein anderes Programm die Ports benutzt. Nach einem Fehler wird das betroffene Modul geschlossen und nicht erneut versucht; das andere Modul wird trotzdem geprüft. Vorherige unvollständige Lesezeilen werden zu Beginn mit CR/LF abgeschlossen. Nicht nach unterbrochenen Schreibbefehlen benutzen.

Auch .NET nutzt denselben Windows-/USB-Treiber und dieselbe Hardware: Fehler hier würden einen ausschließlichen Fehler in unserem Python-Code weniger wahrscheinlich machen, beweisen aber keinen Hardwaredefekt. Es handelt sich nicht um das Herstellerprogramm iseg Terminal.

In [17]:
# Windows/.NET comparison; no additional software installation required.
# Run only the bundled read diagnostic, not the NHS driver.
import base64
import json
import subprocess
from pathlib import Path
from serial.tools import list_ports
from IPython.display import display

native_cycles = 5
if type(native_cycles) is not int or not 1 <= native_cycles <= 20:
    raise ValueError("native_cycles muss zwischen 1 und 20 liegen")

import sys
candidates = [Path.cwd(), *Path.cwd().parents, Path.cwd() / "LAPIS-Control-Suite"]
project_root = next((p.resolve() for p in candidates
                     if (p / "pyproject.toml").is_file()
                     and (p / "src/lapis_control").is_dir()), None)
if project_root is None:
    raise FileNotFoundError("Open this notebook from the LAPIS project checkout.")
src_dir = project_root / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))
script_path = src_dir / "lapis_control/devices/bender/tests/test_usb_dotnet.ps1"
if not script_path.is_file():
    raise FileNotFoundError("Bundled tests/test_usb_dotnet.ps1 was not found.")

ports = list(list_ports.comports())  # Enumerate ports only; communication uses .NET.
def native_find_port(usb_serial):
    matches = [p.device for p in ports if p.serial_number == usb_serial]
    if len(matches) != 1:
        raise RuntimeError(f"USB-Seriennummer {usb_serial}: Ports {matches!r}")
    port = matches[0]
    if not port.upper().startswith("COM") or not port[3:].isdigit():
        raise ValueError(f"Unerwarteter Windows-Port: {port!r}")
    return port

native_positive_port = native_find_port("AI05M8IUA")
native_negative_port = native_find_port("A600C0IIA")
print(f".NET-Lesetest: positiv {native_positive_port}, negativ {native_negative_port}")
# EncodedCommand preserves the encoding of the local test script;
# no execution policy or system setting is changed.
script = script_path.read_text(encoding="utf-8")
command = (
    "[Console]::OutputEncoding = [System.Text.UTF8Encoding]::new($false)\n"
    "& {\n" + script + "\n} "
    f"-PositivePort '{native_positive_port}' -NegativePort '{native_negative_port}' "
    f"-Cycles {native_cycles}\n"
)
encoded = base64.b64encode(command.encode("utf-16-le")).decode("ascii")
process = subprocess.run(
    ["powershell.exe", "-NoProfile", "-NonInteractive", "-EncodedCommand", encoded],
    capture_output=True, encoding="utf-8-sig", errors="replace", timeout=45,
    creationflags=subprocess.CREATE_NO_WINDOW,
)
if process.returncode:
    raise RuntimeError(f"PowerShell fehlgeschlagen:\n{process.stderr}\n{process.stdout}")
native_usb_test = json.loads(process.stdout)
print(native_usb_test["transport"])
for name, port in [("positive", native_positive_port), ("negative", native_negative_port)]:
    rows = [row for row in native_usb_test["results"] if row["module"] == name]
    successful = [row for row in rows if row["ok"]]
    failures = [row for row in rows if not row["ok"]]
    complete = sum(
        sum(row["ok"] and row["cycle"] == cycle for row in rows) == 6
        for cycle in range(1, native_cycles + 1)
    )
    print(f"\n{name}, {port}: {complete}/{native_cycles} volle Zyklen, "
          f"{len(successful)} erfolgreiche Leseabfragen, {len(failures)} Fehler")
    if failures:
        display(failures)
        print("Letzte Rohdaten dieses Ports:")
        trace = [event for event in native_usb_test["trace"] if event["port"] == port]
        for event in trace[-8:]:
            print(event["direction"], repr(event["text"]), "HEX:", event["hex"])
print("\nTest beendet; Verbindungen geschlossen. Notebook mit Strg+S speichern.")
# Full raw trace if needed: display(native_usb_test["trace"])


.NET-Lesetest: positiv COM12, negativ COM11
Windows .NET System.IO.Ports.SerialPort

positive, COM12: 5/5 volle Zyklen, 30 erfolgreiche Leseabfragen, 0 Fehler

negative, COM11: 5/5 volle Zyklen, 30 erfolgreiche Leseabfragen, 0 Fehler

Test beendet; Verbindungen geschlossen. Notebook mit Strg+S speichern.


## 2. Verbindung und Modulstatus lesen

**Treiberkorrektur:** Die Timeout-Zuweisungen während laufender Übertragungen wurden entfernt. In der installierten Windows-Version von pyserial rufen sie `SetCommState` auf und konfigurieren den COM-Port neu. Mit unveränderten Port-Einstellungen liefen anschließend zehn vollständige Lesezyklen an beiden Modulen einschließlich aller vier Bender-Kanäle fehlerfrei. Die damals 26 Offline-Tests bestanden.

Den **Kernel neu starten**, dann die Konfigurationszelle und Punkte 2, 3 und 4 erneut ausführen. Die erhaltenen Ausgaben zeigen teilweise noch den Lauf vor dieser Korrektur. Blockversand bleibt aktiv. Bei Firmware 2.20 wird die Polarität anhand der vorzeichenbehafteten Nennspannung geprüft; `%` und `%/s` werden für die Rampenantwort akzeptiert. Keine Sollwerte oder Ausgänge werden verändert.

In [6]:
with bender_device as bender:
    for name, module in [("Positiv", bender.positive), ("Negativ", bender.negative)]:
        print(f"{name}: {module.port}, {module.channel_count} Kanäle")
        display(asdict(module.read_module()))

Positiv: COM12, 6 Kanäle


{'identification': 'iseg Spezialelektronik GmbH,NHS 40 205,8000311,2.20',
 'temperature_c': 23.0,
 'status': <ModuleStatus.NO_SUM_ERROR|NO_RAMP|SAFETY_LOOP_GOOD|MODULE_GOOD|SUPPLY_GOOD|TEMPERATURE_GOOD: 30464>,
 'events': <ModuleEvent: 0>,
 'voltage_ramp_percent_s': 1.0,
 'timestamp': 1795714694.187068}

Negativ: COM11, 6 Kanäle


{'identification': 'iseg Spezialelektronik GmbH,NHS 40 205,8000312,2.20',
 'temperature_c': 21.4,
 'status': <ModuleStatus.NO_SUM_ERROR|NO_RAMP|SAFETY_LOOP_GOOD|MODULE_GOOD|SUPPLY_GOOD|TEMPERATURE_GOOD: 30464>,
 'events': <ModuleEvent: 0>,
 'voltage_ramp_percent_s': 1.0,
 'timestamp': 1795714694.3538156}

## 2a. Diagnose bei widersprüchlicher Polaritätsmeldung

Nur zur Fehlersuche statt Punkt 2 ausführen. COM12 bleibt als positiv und COM11 als negativ zugeordnet. Diese Zelle öffnet die Ports direkt und liest Gerätekennungen, Kanalstatus sowie rohe Spannungsantworten. Sie ändert weder Sollwerte noch Ausgänge und lockert keine Prüfung im Treiber. Die Spannungsantworten bleiben unverändert; sie werden nicht mit einem konfigurierten Vorzeichen versehen. Werte von null oder ohne Vorzeichen beweisen keine Hardwarepolarität.

Die Geräte-Seriennummer ist das dritte Feld der Gerätekennung (*IDN?); sie ist von der USB-Seriennummer zu unterscheiden. Die vollständige Ausgabe nach dem Ausführen speichern und die Gerätekennung mit dem Typenschild vergleichen.

Für die Kommunikationsdiagnose Punkt 1a verwenden. Diese Diagnose verwendet weiterhin den Blockversand. Beschädigte Echos führen zum Abbruch; sie sind keine Messwerte.

In [7]:
# Documented read queries only; no setpoint or output commands.
# Direct serial access for diagnosing failures during the polarity check.
import serial
import time
from serial.tools import list_ports


def diagnostic_query(connection, command, timeout=2.0):
    wire = (command + "\r\n").encode("ascii")
    print(f"  TX {wire!r}")
    if connection.write(wire) != len(wire):
        raise RuntimeError("Unvollständiger Schreibvorgang bei Leseabfrage.")
    deadline = time.monotonic() + timeout
    while time.monotonic() < deadline:
        raw = connection.read_until(b"\r\n", 202)
        print(f"  RX {raw!r}")
        if not raw.endswith(b"\r\n") or len(raw) > 200:
            raise RuntimeError(f"Keine vollständige Antwort auf {command!r}.")
        answer = raw[:-2].decode("ascii").strip()
        if len(command) > 1 and answer != command and answer.endswith(command[1:]):
            raise RuntimeError(f"Beschaedigtes Echo: {answer!r}; Verbindung wird geschlossen.")
        if answer and answer != command:
            return answer
    raise TimeoutError(f"Keine Antwort auf {command!r}.")


usb_ports = {p.device.casefold(): p for p in list_ports.comports()}
for port_name, expected_polarity in [("COM12", "+"), ("COM11", "-")]:
    print(f"\n=== {port_name}: laut Aufbau {expected_polarity} ===")
    usb_port = usb_ports.get(port_name.casefold())
    print("USB-Seriennummer:", usb_port.serial_number if usb_port else "Port nicht gefunden")
    try:
        with serial.Serial(
            port=port_name, baudrate=9600, bytesize=8, parity="N", stopbits=1,
            timeout=2.0, write_timeout=2.0,
            xonxoff=False, rtscts=False, dsrdtr=False,
        ) as connection:
            connection.reset_input_buffer()
            print("Gerätekennung:", diagnostic_query(connection, "*IDN?"))
            print("Firmware:", diagnostic_query(connection, ":READ:FIRMWARE:NAME?"))
            count = int(diagnostic_query(connection, ":READ:MODULE:CHANNELNUMBER?"))
            if not 1 <= count <= 6:
                raise RuntimeError(f"Unerwartete NHS-Kanalzahl: {count}")
            for ch in range(count):
                status = int(diagnostic_query(connection, f":READ:CHAN:STAT? (@{ch})"))
                v_nom = diagnostic_query(connection, f":READ:VOLT:NOM? (@{ch})")
                v_set = diagnostic_query(connection, f":READ:VOLT? (@{ch})")
                v_meas = diagnostic_query(connection, f":MEAS:VOLT? (@{ch})")
                print(
                    f"  Kanal {ch}: Status={status} / 0x{status:04X}, "
                    f"Bit0={status & 1}, ON={bool(status & 8)}, "
                    f"VNOM={v_nom!r}, VSET={v_set!r}, VMEAS={v_meas!r}"
                )
    except Exception as exc:
        # Inspect the second port even if the first one fails.
        print(f"DIAGNOSEFEHLER {port_name}: {type(exc).__name__}: {exc}")


=== COM12: laut Aufbau + ===
USB-Seriennummer: AI05M8IUA
  TX b'*IDN?\r\n'
  RX b'*IDN?\r\n'
  RX b'iseg Spezialelektronik GmbH,NHS 40 205,8000311,2.20\r\n'
Gerätekennung: iseg Spezialelektronik GmbH,NHS 40 205,8000311,2.20
  TX b':READ:FIRMWARE:NAME?\r\n'
  RX b':READ:FIRMWARE:NAME?\r\n'
  RX b'N06C2\r\n'
Firmware: N06C2
  TX b':READ:MODULE:CHANNELNUMBER?\r\n'
  RX b':READ:MODULE:CHANNELNUMBER?\r\n'
  RX b'6\r\n'
  TX b':READ:CHAN:STAT? (@0)\r\n'
  RX b':READ:CHAN:STAT? (@0)\r\n'
  RX b'0\r\n'
  TX b':READ:VOLT:NOM? (@0)\r\n'
  RX b':READ:VOLT:NOM? (@0)\r\n'
  RX b'4.00000E3V\r\n'
  TX b':READ:VOLT? (@0)\r\n'
  RX b':READ:VOLT? (@0)\r\n'
  RX b'0.01000E3V\r\n'
  TX b':MEAS:VOLT? (@0)\r\n'
  RX b':MEAS:VOLT? (@0)\r\n'
  RX b'-0.00005E3V\r\n'
  Kanal 0: Status=0 / 0x0000, Bit0=0, ON=False, VNOM='4.00000E3V', VSET='0.01000E3V', VMEAS='-0.00005E3V'
  TX b':READ:CHAN:STAT? (@1)\r\n'
  RX b':READ:CHAN:STAT? (@1)\r\n'
  RX b'0\r\n'
  TX b':READ:VOLT:NOM? (@1)\r\n'
  RX b':READ:VOLT:NOM? (@1

## 3. Bender-Kanäle lesen
Spannungen in V mit Vorzeichen, Ströme in A, Zeitstempel als Unixzeit. Die Messungen erfolgen nacheinander.

In [18]:
with bender_device as bender:
    show_channels(bender)

[{'name': 'p0',
  'channel': 0,
  'voltage_set': 10.0,
  'voltage': 0.05,
  'current_set': 0.002,
  'current': -1.1e-07,
  'status': '0x0000 (<ChannelStatus: 0>)',
  'events': '0x0090 (<ChannelEvent.END_OF_VOLTAGE_RAMP|CONSTANT_VOLTAGE: 144>)',
  'timestamp': 1795715676.1166472},
 {'name': 'p1',
  'channel': 1,
  'voltage_set': 10.0,
  'voltage': 0.17,
  'current_set': 0.002,
  'current': -1.3e-07,
  'status': '0x0000 (<ChannelStatus: 0>)',
  'events': '0x0090 (<ChannelEvent.END_OF_VOLTAGE_RAMP|CONSTANT_VOLTAGE: 144>)',
  'timestamp': 1795715676.3882558},
 {'name': 'n0',
  'channel': 0,
  'voltage_set': -10.0,
  'voltage': -0.05,
  'current_set': 0.002,
  'current': 1.2e-07,
  'status': '0x0000 (<ChannelStatus: 0>)',
  'events': '0x0090 (<ChannelEvent.END_OF_VOLTAGE_RAMP|CONSTANT_VOLTAGE: 144>)',
  'timestamp': 1795715676.655635},
 {'name': 'n1',
  'channel': 1,
  'voltage_set': -10.0,
  'voltage': -0.13,
  'current_set': 0.002,
  'current': -0.0,
  'status': '0x0000 (<ChannelStatus: 0

## 4. Kurze Messreihe
Zehn Lesezyklen. Bei Kernel-Unterbrechung wird die Verbindung geschlossen.

In [19]:
measurements = []
with bender_device as bender:
    for sample in range(10):
        readings = bender.read_all()
        measurements.extend({"name": name, **asdict(r)} for name, r in readings.items())
        print(sample + 1, " | ".join(
            f"{name}: {r.voltage:.3f} V, {r.current:.3e} A"
            for name, r in readings.items()))
        time.sleep(0.5)

1 p0: 0.050 V, -1.200e-07 A | p1: 0.190 V, -1.100e-07 A | n0: -0.050 V, 1.000e-07 A | n1: -0.130 V, -0.000e+00 A
2 p0: 0.050 V, -1.300e-07 A | p1: 0.150 V, -1.200e-07 A | n0: -0.050 V, 1.100e-07 A | n1: -0.130 V, -0.000e+00 A
3 p0: 0.050 V, -1.300e-07 A | p1: 0.180 V, -1.300e-07 A | n0: -0.050 V, 1.200e-07 A | n1: -0.130 V, -0.000e+00 A
4 p0: 0.050 V, -1.300e-07 A | p1: 0.200 V, -1.100e-07 A | n0: -0.050 V, 1.200e-07 A | n1: -0.130 V, -0.000e+00 A
5 p0: 0.050 V, -1.300e-07 A | p1: 0.190 V, -1.000e-07 A | n0: -0.050 V, 1.200e-07 A | n1: -0.120 V, -0.000e+00 A
6 p0: 0.050 V, -1.200e-07 A | p1: 0.200 V, -1.000e-07 A | n0: -0.050 V, 1.200e-07 A | n1: -0.110 V, -0.000e+00 A
7 p0: 0.050 V, -1.400e-07 A | p1: 0.140 V, -1.200e-07 A | n0: -0.050 V, 1.300e-07 A | n1: -0.120 V, -0.000e+00 A
8 p0: 0.050 V, -1.200e-07 A | p1: 0.160 V, -1.300e-07 A | n0: -0.050 V, 1.300e-07 A | n1: -0.110 V, -0.000e+00 A
9 p0: 0.050 V, -1.100e-07 A | p1: 0.180 V, -1.100e-07 A | n0: -0.050 V, 1.300e-07 A | n1: -0.110

## 5. Optional: Sollwerte setzen
Nach Prüfung des Aufbaus ALLOW_WRITES in der Konfiguration auf True setzen und diese erneut ausführen. Beispielwerte vorher anpassen. Die vier Bender-Kanäle müssen ausgeschaltet sein. Sollwerte bleiben erhalten; Änderungen erfolgen nacheinander und können teilweise fehlschlagen.

In [13]:
test_voltages = dict(p0=10.0, p1=10.0, n0=-10.0, n1=-10.0)
if ALLOW_WRITES:
    with bender_device as bender:
        if any(r.status & ChannelStatus.ON for r in bender.read_all().values()):
            raise RuntimeError("Vor dem Sollwerttest alle vier Bender-Kanäle ausschalten.")
        bender.set_voltages(**test_voltages)
        show_channels(bender)
else:
    print("Übersprungen: ALLOW_WRITES ist False.")

[{'name': 'p0',
  'channel': 0,
  'voltage_set': 10.0,
  'voltage': 0.05,
  'current_set': 0.002,
  'current': -1.6e-07,
  'status': '0x0000 (<ChannelStatus: 0>)',
  'events': '0x0090 (<ChannelEvent.END_OF_VOLTAGE_RAMP|CONSTANT_VOLTAGE: 144>)',
  'timestamp': 1794761328.7412329},
 {'name': 'p1',
  'channel': 1,
  'voltage_set': 10.0,
  'voltage': 0.21,
  'current_set': 0.002,
  'current': -1.3e-07,
  'status': '0x0000 (<ChannelStatus: 0>)',
  'events': '0x0090 (<ChannelEvent.END_OF_VOLTAGE_RAMP|CONSTANT_VOLTAGE: 144>)',
  'timestamp': 1794761329.0129647},
 {'name': 'n0',
  'channel': 0,
  'voltage_set': -10.0,
  'voltage': -0.05,
  'current_set': 0.002,
  'current': 1e-07,
  'status': '0x0000 (<ChannelStatus: 0>)',
  'events': '0x0090 (<ChannelEvent.END_OF_VOLTAGE_RAMP|CONSTANT_VOLTAGE: 144>)',
  'timestamp': 1794761329.276217},
 {'name': 'n1',
  'channel': 1,
  'voltage_set': -10.0,
  'voltage': -0.14,
  'current_set': 0.002,
  'current': -0.0,
  'status': '0x0000 (<ChannelStatus: 0>)

## 6. Optional: Hochspannung für fünf Sekunden einschalten
Zusätzlich ALLOW_HV_ON freigeben. Verwendet die am Gerät vorhandenen Sollwerte: vorher Zelle 3 prüfen. Anschließend und bei Fehlern wird das Ausschalten aller vier Bender-Kanäle versucht. Kommunikationsfehler können das Ausschalten verhindern. Ausschalten fordert ein Abfahren an; Restspannung anschließend messen. Kanäle 2–5 werden nicht geschaltet.

In [12]:
if ALLOW_WRITES and ALLOW_HV_ON:
    with bender_device as bender:
        try:
            bender.enable()
            time.sleep(5.0)
            show_channels(bender)
        finally:
            bender.disable()
else:
    print("Übersprungen: ALLOW_WRITES und ALLOW_HV_ON müssen True sein.")

[{'name': 'p0',
  'channel': 0,
  'voltage_set': 10.0,
  'voltage': 8.78,
  'current_set': 0.002,
  'current': -1e-08,
  'status': '0x0088 (<ChannelStatus.ON|CONSTANT_VOLTAGE: 136>)',
  'events': '0x0090 (<ChannelEvent.END_OF_VOLTAGE_RAMP|CONSTANT_VOLTAGE: 144>)',
  'timestamp': 1795714787.4267433},
 {'name': 'p1',
  'channel': 1,
  'voltage_set': 10.0,
  'voltage': 8.79,
  'current_set': 0.002,
  'current': -0.0,
  'status': '0x0088 (<ChannelStatus.ON|CONSTANT_VOLTAGE: 136>)',
  'events': '0x0090 (<ChannelEvent.END_OF_VOLTAGE_RAMP|CONSTANT_VOLTAGE: 144>)',
  'timestamp': 1795714787.714853},
 {'name': 'n0',
  'channel': 0,
  'voltage_set': -10.0,
  'voltage': -9.96,
  'current_set': 0.002,
  'current': 4e-08,
  'status': '0x0088 (<ChannelStatus.ON|CONSTANT_VOLTAGE: 136>)',
  'events': '0x0090 (<ChannelEvent.END_OF_VOLTAGE_RAMP|CONSTANT_VOLTAGE: 144>)',
  'timestamp': 1795714787.981443},
 {'name': 'n1',
  'channel': 1,
  'voltage_set': -10.0,
  'voltage': -9.84,
  'current_set': 0.002,


## 6a. Einkanalversuch mit +15 V auf p0

Für diesen Versuch nur `ALLOW_P0_TEST` in der Konfigurationszelle freigeben und anschließend diese Zelle ausführen. Ausgangsfreigabe und Aufbau vor Ort prüfen. Der Versuch öffnet das positive Modul und benötigt einen ausgeschalteten p0. Das negative Modul wird nicht geöffnet.

Der Versuch erfasst zwölf interne Messwerte im Abstand von mindestens fünf Sekunden. Die letzte Spalte zeigt den neu angeforderten Sollwert. Als stabiles Ergebnis zählen hier drei aufeinanderfolgende Messungen am Ende innerhalb von 15,0 ± 0,2 V nach Rampenende. Für den Vergleich an der Elektrode den externen Messwert unter Betriebslast mit dem zugehörigen Zeitpunkt notieren. Das interne Toleranzband beschreibt die Regelung auf die Geräteanzeige.

Für p0 sind Sollwerte von 10 bis 100 V zugelassen. Ein kleineres Anwendungslimit aus der Konfiguration hat Vorrang. Meldet das Modul eine Spannung über der wirksamen Obergrenze, bricht die Regelung bei dieser Abfrage ab. Am Ende und bei Abbruch fordert die Zelle das Herunterfahren von p0 an. Nach bestätigtem Aus-Zustand und weniger als 1 V internem Restspannungsbetrag stellt sie den vorherigen Sollwert wieder her, sofern er innerhalb des Anwendungslimits und höchstens bei 100 V liegt. Bei einem Verbindungsfehler kann der Ausgangszustand ungeklärt sein.

Im ersten Hardwareversuch am 26.11.2026 um 18:28 Uhr mit 20 V Stellgrenze meldete p0 nach der ersten Korrektur 30,03 V. Die unten gespeicherte Ausgabe zeigt den späteren Lauf mit 100 V Obergrenze. Von 43,4 bis 64,1 s meldete p0 bei 15,12 V Stellwert fünf Werte zwischen 15,12 und 15,13 V. Beim späteren [Multimetervergleich am 28.11.2026](README.md#multimetervergleich) stimmten die NHS-Messwerte an den dort geprüften Punkten gut mit der Multimeteranzeige überein.

In [20]:
from itertools import islice
from lapis_control.devices.bender.iseg_nhs import ModuleStatus

if ALLOW_P0_TEST:
    p0_samples = []
    with bender_device.positive as module:
        initial = module.read_channel(0)
        module_status = module.read_module_status()
        if initial.status & ~(ChannelStatus.POSITIVE | ChannelStatus.LOW_CURRENT_RANGE):
            raise RuntimeError(f"p0 vor dem Start prüfen. Kanalstatus {initial.status!r}")
        if abs(initial.voltage) >= 1.0 or not module_status & ModuleStatus.MODULE_GOOD:
            raise RuntimeError(f"Startzustand prüfen. p0 {initial.voltage:g} V, Modulstatus {module_status!r}")
        print(f"{module.identification} | p0 vor dem Versuch: VSET {initial.voltage_set:.2f} V, "
              f"VMEAS {initial.voltage:.2f} V")
        target_v, tolerance_v = 15.0, 0.2
        p0_max_v = min(100.0, module.max_voltage)
        module.set_voltage(0, target_v)
        try:
            module.enable(0)
            started = time.monotonic()
            print("Zeit [s]   VSET [V]   VMEAS [V]   neuer VSET [V]")
            feedback = module.regulate_voltage(0, target_v, tolerance_v=tolerance_v,
                                               max_step_v=1.0, setpoint_range_v=(10.0, p0_max_v), interval_s=5.0)
            for sample in islice(feedback, 12):
                p0_samples.append(sample)
                command = "–" if sample.requested_voltage is None else f"{sample.requested_voltage:.2f}"
                print(f"{time.monotonic()-started:8.1f}   {sample.voltage_set:8.2f}   "
                      f"{sample.voltage:9.2f}   {command:>14}")
            stable = all(s.in_tolerance for s in p0_samples[-3:])
            print("Letzte drei Messungen im Zielband:", stable)
        finally:
            module.disable(0)
            deadline = time.monotonic() + 10.0
            while True:
                final = module.read_channel(0)
                if not final.status & (ChannelStatus.ON | ChannelStatus.RAMPING) and abs(final.voltage) < 1.0:
                    break
                if time.monotonic() >= deadline:
                    raise RuntimeError(f"Abfahren von p0 nicht bestätigt: {final!r}")
                time.sleep(0.5)
            print(f"p0 aus, interne Restspannung {final.voltage:.2f} V.")
            if abs(initial.voltage_set) <= p0_max_v:
                module.set_voltage(0, initial.voltage_set)
                print(f"Vorheriger Sollwert bei ausgeschaltetem p0 bestätigt: {initial.voltage_set:.2f} V.")
            else:
                print(f"Letzter Versuchssollwert bleibt gespeichert: {final.voltage_set:.2f} V.")
else:
    print("Einkanalversuch übersprungen. ALLOW_P0_TEST ist False.")

iseg Spezialelektronik GmbH,NHS 40 205,8000311,2.20 | p0 vor dem Versuch: VSET 10.00 V, VMEAS 0.05 V
Zeit [s]   VSET [V]   VMEAS [V]   neuer VSET [V]
     5.6      15.00       13.77            15.62
    11.1      15.61       14.38            15.92
    16.6      15.92       14.72            16.06
    21.8      16.06       14.84                –
    27.3      16.06       16.05            15.53
    32.8      15.53       15.53            15.27
    38.3      15.26       15.27            15.12
    43.4      15.12       15.12                –
    48.6      15.12       15.13                –
    53.7      15.12       15.12                –
    58.9      15.12       15.13                –
    64.1      15.12       15.12                –
Letzte drei Messungen im Zielband: True
p0 aus, interne Restspannung 0.15 V.
Vorheriger Sollwert bei ausgeschaltetem p0 bestätigt: 10.00 V.


## 6b. Vier Elektroden auf demselben Spannungsbetrag halten

Für diesen Test genügen die Konfigurationszelle und Abschnitt 6b. In der Konfigurationszelle `ALLOW_FOUR_CHANNEL_TEST = True` setzen. Die übrigen Testzellen haben eigene Freigaben. Nach einem Treiberupdate den Kernel neu starten. Alle vier Bender-Ausgänge müssen anfangs aus sein und intern weniger als 1 V anzeigen.

Für eine andere Spannung genügt es, `TARGET_MAGNITUDE_V` zu ändern. Bei 10 V erhalten p0/p1 den Zielwert +10 V und n0/n1 den Zielwert −10 V. Der erlaubte Stellbetrag reicht von 0 bis zum Doppelten des Zielbetrags, höchstens 100 V bzw. dem kleineren konfigurierten Modullimit. Das ergibt 0–20 V bei 10 V Ziel oder 0–30 V bei 15 V Ziel. Der Messwertabbruch verwendet das gemeinsame absolute Limit.

Die Laufzeit `DURATION_S` beginnt, sobald alle vier Kanäle eingeschaltet sind. In den ersten `INITIAL_SETTLE_S` (Standard 20 s) werden alle 5 s Spannungen und Status geprüft. Die NHS-Module halten dabei den ursprünglichen Stellwert. Erst danach kann die Software bei beendeter Rampe außerhalb von ±0,2 V korrigieren, um die halbe Abweichung und höchstens 1 V je Schritt. Drei aufeinanderfolgende Treffer nach der Einschwingzeit schalten den Kanal auf den 30-s-Haltetakt um. Bei Drift oder Rampe gilt wieder der kurze Takt. Im gespeicherten 120-s-Lauf bei ±15 V erreichten alle vier Kanäle den Haltetakt ohne Stellkorrektur.

Die Tabelle zeigt intern gemessene Spannungen. „Halten“ bezeichnet den Takt bis zur nächsten Prüfung dieser Elektrode. Die vier Zeitpunkte können sich durch die serielle Übertragung unterscheiden. Am Ende, bei Unterbrechung oder bei einem Fehler fordert der Ablauf das Abfahren aller vier Kanäle an. Nach bestätigtem Aus-Zustand, Rampenende und weniger als 1 V Messspannung stellt er die zuvor gespeicherten zulässigen Sollwerte wieder her.

In [2]:
if ALLOW_FOUR_CHANNEL_TEST:
    from lapis_control.workflows.bender_voltage import hold_bender_voltage

    TARGET_MAGNITUDE_V = 15.0
    DURATION_S = 120.0
    INITIAL_SETTLE_S = 20.0
    SETTLE_INTERVAL_S = 5.0
    HOLD_INTERVAL_S = 30.0
    four_channel_samples = []

    def report_feedback(key, sample, holding):
        elapsed = time.monotonic() - trial_start
        four_channel_samples.append((elapsed, key, sample, holding))
        command = "–" if sample.requested_voltage is None else f"{sample.requested_voltage:+.2f}"
        phase = "Halten" if holding else "Einregeln"
        if sample.observing:
            phase = "Einschwingen"
        print(f"{elapsed:7.1f}  {key:>5}  {sample.voltage_set:+8.2f}  {sample.voltage:+9.2f}"
              f"  {command:>14}  {phase}")

    with bender_device as bender:
        trial_start = time.monotonic()
        print("Zeit [s]  Kanal  VSET [V]  VMEAS [V]  neuer VSET [V]  nächste Prüfphase")
        stable = hold_bender_voltage(bender, TARGET_MAGNITUDE_V, duration_s=DURATION_S,
                                     initial_settle_s=INITIAL_SETTLE_S,
                                     settle_interval_s=SETTLE_INTERVAL_S, hold_interval_s=HOLD_INTERVAL_S,
                                     on_sample=report_feedback)
    print("Alle vier Kanäle bei ihrer letzten Prüfung stabil:", stable)
    for key in bender_device.channels:
        samples = [sample for _, name, sample, _ in four_channel_samples if name == key]
        writes = sum(s.requested_voltage is not None for s in samples)
        print(f"{key}: {len(samples)} Messungen, {writes} Korrekturen")
    print("Alle vier Kanäle aus, internes Abfahrkriterium bestätigt.")
else:
    print("Vierkanalversuch übersprungen. ALLOW_FOUR_CHANNEL_TEST ist False.")

Zeit [s]  Kanal  VSET [V]  VMEAS [V]  neuer VSET [V]  nächste Prüfphase
   10.5     p0    +15.00     +13.74               –  Einschwingen
   10.8     p1    +15.00     +13.82               –  Einschwingen
   11.1     n0    -15.00     -14.99               –  Einschwingen
   11.3     n1    -15.00     -14.95               –  Einschwingen
   15.6     p0    +15.00     +13.73               –  Einschwingen
   15.9     p1    +15.00     +13.75               –  Einschwingen
   16.2     n0    -15.00     -14.96               –  Einschwingen
   16.5     n1    -15.00     -14.94               –  Einschwingen
   20.8     p0    +15.00     +15.02               –  Einschwingen
   21.1     p1    +15.00     +15.00               –  Einschwingen
   21.4     n0    -15.00     -14.89               –  Einschwingen
   21.6     n1    -15.00     -14.93               –  Einschwingen
   26.0     p0    +15.00     +15.01               –  Einregeln
   26.2     p1    +15.00     +14.97               –  Einregeln
   26.5   

## 6c. n0 bei festem −15-V-Sollwert beobachten und anschließend nachregeln

Nur die Konfigurationszelle und diese Zelle ausführen. Die separate Freigabe ist `ALLOW_N0_DIAGNOSTIC`. Der Test verwendet Kanal 0 des negativen Moduls (Seriennummer 8000312). Er erfasst 60 s lang den Sollwert und die interne Messspannung bei fest programmierten −15 V. Die letzte Spalte der Festwertphase enthält die originale SCPI-Messantwort für dieselbe Abfrage.

Erst danach darf die Regelung den Sollwert alle fünf Sekunden um höchstens 1 V korrigieren. Der Stellbetrag liegt zwischen 10 und 100 V und innerhalb des Anwendungslimits. Drei Messwerte innerhalb von −15,0 ± 0,2 V bei beendeter Rampe beenden den Versuch. Nach spätestens zwölf Regelmessungen oder bei Abbruch fährt n0 herunter. Der Test prüft den Aus-Zustand und stellt den anfänglichen Sollwert wieder her. Die Spannungen stammen aus dem NHS.

In [ ]:
from datetime import datetime
from itertools import islice
from lapis_control.devices.bender.iseg_nhs import VerificationError, parse_number

if ALLOW_N0_DIAGNOSTIC:
    target_v, tolerance_v = -15.0, 0.2
    n0_fixed, n0_feedback = [], []
    with bender_device.negative as module:
        if module.polarity != "-" or module.expected_serial != "8000312":
            raise RuntimeError("Für diesen Aufbau wird das negative NHS 8000312 erwartet.")
        module.check_regulation_module_status()
        initial = module.read_channel(0)
        if initial.status & ~ChannelStatus.LOW_CURRENT_RANGE or abs(initial.voltage) >= 1.0:
            raise RuntimeError(f"n0 vor dem Start prüfen: {initial!r}")
        if abs(initial.voltage_set) > min(100.0, module.max_voltage):
            raise RuntimeError("Der ursprüngliche Sollwert liegt außerhalb des wiederherstellbaren Bereichs.")
        print(datetime.now().astimezone().isoformat(timespec="seconds"), module.identification)
        print(f"n0 vor dem Versuch: VSET {initial.voltage_set:+.2f} V, VMEAS {initial.voltage:+.2f} V")
        try:
            module.set_voltage(0, target_v)
            module.enable(0)
            start = time.monotonic()
            print("Festwert: Zeit [s] | VSET [V] | VMEAS [V] | Status | rohe NHS-Messantwort")
            for sample_index in range(31):
                time.sleep(max(0.0, start + 2.0 * sample_index - time.monotonic()))
                vset = module.read_voltage_set(0)
                raw = module.query(":MEAS:VOLT? (@0)")
                elapsed = time.monotonic() - start
                measured = module.sign * abs(parse_number(raw, "V"))
                channel_status = module.read_status(0)
                n0_fixed.append((elapsed, vset, measured, channel_status, raw))
                print(f"{elapsed:6.1f} {vset:+9.2f} {measured:+10.2f}    {int(channel_status):#06x}   {raw}")
                module.check_regulation_module_status()
                allowed = (ChannelStatus.ON | ChannelStatus.CONSTANT_VOLTAGE |
                           ChannelStatus.RAMPING | ChannelStatus.LOW_CURRENT_RANGE)
                if (channel_status & ~allowed or not channel_status & ChannelStatus.ON
                        or not channel_status & (ChannelStatus.RAMPING | ChannelStatus.CONSTANT_VOLTAGE)):
                    raise RuntimeError(f"n0: unerwarteter Kanalstatus {channel_status!r}")
                if abs(vset - target_v) > 0.01 or abs(measured) > min(100.0, module.max_voltage):
                    raise VerificationError(f"Unexpected n0 fixed-phase readback: VSET {vset:g}, VMEAS {measured:g}")
            print("Regelung: Zeit [s] | VSET [V] | VMEAS [V] | neuer VSET [V]")
            stable_count = 0
            feedback = module.regulate_voltage(0, target_v, tolerance_v=tolerance_v, max_step_v=1.0,
                                               setpoint_range_v=(10.0, 100.0), interval_s=5.0,
                                               convergence_timeout_s=60.0)
            for sample in islice(feedback, 12):
                elapsed = time.monotonic() - start
                n0_feedback.append((elapsed, sample))
                command = "–" if sample.requested_voltage is None else f"{sample.requested_voltage:+.2f}"
                print(f"{elapsed:6.1f} {sample.voltage_set:+9.2f} {sample.voltage:+10.2f}   {command:>8}")
                stable_count = stable_count + 1 if sample.in_tolerance else 0
                if stable_count == 3:
                    break
            print("Drei aufeinanderfolgende interne Messungen im Zielband:", stable_count == 3)
        finally:
            module.disable(0)
            deadline = time.monotonic() + 10.0
            while True:
                final = module.read_channel(0)
                if not final.status & (ChannelStatus.ON | ChannelStatus.RAMPING) and abs(final.voltage) < 1.0:
                    break
                if time.monotonic() >= deadline:
                    raise RuntimeError(f"Abfahren von n0 nicht bestätigt: {final!r}")
                time.sleep(0.5)
            module.set_voltage(0, initial.voltage_set)
            restored = module.read_channel(0)
            if (restored.status & (ChannelStatus.ON | ChannelStatus.RAMPING) or abs(restored.voltage) >= 1.0
                    or abs(restored.voltage_set - initial.voltage_set) > 0.01):
                raise RuntimeError(f"Endzustand von n0 prüfen: {restored!r}")
            print(f"n0 aus, ursprünglicher VSET {restored.voltage_set:+.2f} V bestätigt, "
                  f"VMEAS {restored.voltage:+.2f} V, Status {int(restored.status):#06x}.")
else:
    print("n0-Diagnose übersprungen. ALLOW_N0_DIAGNOSTIC ist False.")

2026-11-26T19:04:04+01:00 iseg Spezialelektronik GmbH,NHS 40 205,8000312,2.20
n0 vor dem Versuch: VSET -10.00 V, VMEAS -0.05 V
Festwert: Zeit [s] | VSET [V] | VMEAS [V] | Status | rohe NHS-Messantwort
   0.1    -15.00     -15.82    0x0088   -0.01582E3V
   2.1    -15.00     -15.02    0x0088   -0.01502E3V
   4.1    -15.00     -14.89    0x0088   -0.01489E3V
   6.1    -15.00     -15.17    0x0088   -0.01517E3V
   8.1    -15.00     -14.94    0x0088   -0.01494E3V
  10.1    -15.00     -14.95    0x0088   -0.01495E3V
  12.1    -15.00     -14.91    0x0088   -0.01491E3V
  14.1    -15.00     -15.32    0x0088   -0.01532E3V
  16.1    -15.00     -15.00    0x0088   -0.01500E3V
  18.1    -15.00     -15.00    0x0088   -0.01500E3V
  20.1    -15.00     -14.88    0x0088   -0.01488E3V
  22.1    -15.00     -15.00    0x0088   -0.01500E3V
  24.1    -15.00     -14.97    0x0088   -0.01497E3V
  26.1    -15.00     -14.98    0x0088   -0.01498E3V
  28.1    -15.00     -14.98    0x0088   -0.01498E3V
  30.1    -15.00   

## 7. Manuell ausschalten und Restspannung lesen
Zum Ausführen die folgenden Zeilen auskommentieren. Wartet nicht auf das Ende der Rampe.

In [ ]:
# with bender_device as bender:
#     bender.disable()
#     show_channels(bender)


[{'name': 'p0',
  'channel': 0,
  'voltage_set': 10.0,
  'voltage': 0.05,
  'current_set': 0.002,
  'current': -1.3e-07,
  'status': '0x0000 (<ChannelStatus: 0>)',
  'events': '0x0090 (<ChannelEvent.END_OF_VOLTAGE_RAMP|CONSTANT_VOLTAGE: 144>)',
  'timestamp': 1794761458.4499884},
 {'name': 'p1',
  'channel': 1,
  'voltage_set': 10.0,
  'voltage': 0.17,
  'current_set': 0.002,
  'current': -1.4e-07,
  'status': '0x0000 (<ChannelStatus: 0>)',
  'events': '0x0090 (<ChannelEvent.END_OF_VOLTAGE_RAMP|CONSTANT_VOLTAGE: 144>)',
  'timestamp': 1794761458.7215807},
 {'name': 'n0',
  'channel': 0,
  'voltage_set': -10.0,
  'voltage': -0.45,
  'current_set': 0.002,
  'current': -0.0,
  'status': '0x0000 (<ChannelStatus: 0>)',
  'events': '0x0090 (<ChannelEvent.END_OF_VOLTAGE_RAMP|CONSTANT_VOLTAGE: 144>)',
  'timestamp': 1794761458.982945},
 {'name': 'n1',
  'channel': 1,
  'voltage_set': -10.0,
  'voltage': -0.04,
  'current_set': 0.002,
  'current': -0.0,
  'status': '0x0000 (<ChannelStatus: 0>)'

## 8. Offline-Treibertests
Simulierte serielle Verbindungen; keine Hardware erforderlich.

In [3]:
import unittest
from lapis_control.devices.bender.tests import test_iseg_nhs

suite = unittest.defaultTestLoader.loadTestsFromModule(test_iseg_nhs)
result = unittest.TextTestRunner(verbosity=2).run(suite)
assert result.wasSuccessful(), "Mindestens ein Offline-Test ist fehlgeschlagen."

test_bad_or_missing_byte_echo_stops_without_retry (lapis_control.devices.bender.tests.test_iseg_nhs.DriverTests.test_bad_or_missing_byte_echo_stops_without_retry) ... ok
test_bender_mapping_enable_and_read (lapis_control.devices.bender.tests.test_iseg_nhs.DriverTests.test_bender_mapping_enable_and_read) ... ok
test_bender_prevalidates_every_voltage (lapis_control.devices.bender.tests.test_iseg_nhs.DriverTests.test_bender_prevalidates_every_voltage) ... ok
test_byte_handshake_consumes_crlf_echo_before_answer (lapis_control.devices.bender.tests.test_iseg_nhs.DriverTests.test_byte_handshake_consumes_crlf_echo_before_answer) ... ok
test_command_length_and_injection (lapis_control.devices.bender.tests.test_iseg_nhs.DriverTests.test_command_length_and_injection) ... ok
test_connect_aliases_reuse_connection_and_reconnect_after_close (lapis_control.devices.bender.tests.test_iseg_nhs.DriverTests.test_connect_aliases_reuse_connection_and_reconnect_after_close) ... ok
test_connect_does_not_reopen